In [1]:
#1. Librerías.
!pip install unidecode
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import ast
import re
import unidecode

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
#2. Constantes.
df_final_path = "./pruebas_batch/df_final_crudo.csv"
df_exportacion_path = "./pruebas_batch/df_final_limpio.csv"

In [3]:
#3. Lectura.
df_final = pd.read_csv(df_final_path)
#pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None) 

/tmp/ipykernel_108958/855688390.py:2: DtypeWarning: Columns (65) have mixed types. Specify dtype option on import or set low_memory=False.
  df_final = pd.read_csv(df_final_path)


In [4]:
#4. El LLM creó en algunos casos columnas incorrectas ----> Reemplazo los valores obtenidos en sus columnas correspondientes.
#a. Reemplazo los valores.
df_final.loc[~df_final["impact_fmi"].isna(),"impacto_fmi"] = df_final.loc[~df_final["impact_fmi"].isna()]["impact_fmi"]
df_final.loc[~df_final["menciona_sector_agroexportador "].isna(),"menciona_sector_agroexportador"] = df_final.loc[~df_final["menciona_sector_agroexportador "].isna()]["menciona_sector_agroexportador "]
df_final.loc[~df_final["activid(ad)_económica"].isna(),"actividad_economica"] = df_final.loc[~df_final["activid(ad)_económica"].isna()]["activid(ad)_económica"]
df_final.loc[~df_final["impact_prestamos_internacionales"].isna(),"impacto_prestamos_internacionales"] = df_final.loc[~df_final["impact_prestamos_internacionales"].isna()]["impact_prestamos_internacionales"]
#b. Elimino las columnas correspondientes.
df_final.drop(columns=["impact_fmi","menciona_sector_agroexportador ","activid(ad)_económica","impact_prestamos_internacionales"], inplace=True)

/tmp/ipykernel_108958/72731286.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[True]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df_final.loc[~df_final["menciona_sector_agroexportador "].isna(),"menciona_sector_agroexportador"] = df_final.loc[~df_final["menciona_sector_agroexportador "].isna()]["menciona_sector_agroexportador "]


In [5]:
#5. Elimino aquellas noticias sin contenido (vacío).
df_final = df_final[~df_final["contenido"].isna()]

In [6]:
#6. Normalización de columnas de "texto libre".
#a. Formateo.
df_final["sectores_mencionados"] = df_final["sectores_mencionados"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

df_final["empresas_mencionadas"] = df_final["empresas_mencionadas"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

df_final["tickers_mencionados"] = df_final["tickers_mencionados"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
#b. Normalizo los sectores.
def normalizar_strings(s):
    s = s.lower().strip() # minúsculas.
    s = unidecode.unidecode(s)  # quita tildes.
    s = s.replace("_", " ").replace("-", " ").replace("/", " ").strip() # reemplazo valores.
    #Eliminar puntos, comas y otros signos (excepto letras, números y espacios)
    s = re.sub(r"[^a-z0-9áéíóúüñ\s]", "", s)
    s = s.replace("(", " ").replace(")", " ")
    return s

def normalizar_sector_macro(string):
    if not isinstance(string, str):
        return string

    s = normalizar_strings(string)

    # Macro-sectores
    if re.search(r'finan|banca|bono|mercado|fondos|deposito|inversion', s):
        return 'Finanzas'
    if re.search(r'cripto|blockchain|nft|defi|fintech', s):
        return 'Cripto / Fintech'
    if re.search(r'tecnolog|software|data|inteligencia|robot|semiconductor|iot|ia', s):
        return 'Tecnología'
    if re.search(r'energ|petroleo|gas|renovable|gnl|electrico|electricidad', s):
        return 'Energía'
    if re.search(r'min', s) and not 'admin' in s:
        return 'Minería'
    if re.search(r'educa|universidad|escuela|formacion|capacita', s):
        return 'Educación'
    if re.search(r'salud|medic|farmaceut|hospital', s):
        return 'Salud'
    if re.search(r'comerc|retail|supermercado|consumo|alimentos|bebid|frigorif', s):
        return 'Comercio / Consumo'
    if re.search(r'transport|logistic|puerto|aero|hidro|movilidad|vehiculos', s):
        return 'Transporte / Logística'
    if re.search(r'gobier|sector publico|administracion|municipal|provincial', s):
        return 'Gobierno / Sector Público'
    if re.search(r'politi|electoral|legislativ|partid', s):
        return 'Política'
    if re.search(r'justic|judicial|tribunal|juridic', s):
        return 'Justicia'
    if re.search(r'segur', s):
        return 'Seguridad'
    if re.search(r'turism|viaje|hotel|balneario|turistico', s):
        return 'Turismo'
    if re.search(r'industri|manufactur|fabrica|metal|quimic|textil|automotriz', s):
        return 'Industria / Manufactura'
    if re.search(r'agro|ganad|rural|agric|cultivo|frut|soja|maiz|trigo', s):
        return 'Agropecuario'
    if re.search(r'inmob|vivienda|constru|obra', s):
        return 'Construcción / Inmobiliario'
    if re.search(r'medio ambiente|ambient|ecolog|forest', s):
        return 'Medio Ambiente'
    if re.search(r'emple|laboral|trabaj', s):
        return 'Empleo / Laboral'
    if re.search(r'servic', s):
        return 'Servicios'
    if re.search(r'infraestr|hidrovía|hidrovia', s):
        return 'Infraestructura'
    
    # Si no entra en ninguna categoría
    return 'Otros'

df_final["sectores_mencionados"] = df_final["sectores_mencionados"].apply(
    lambda lista: list(dict.fromkeys([normalizar_sector_macro(s) for s in lista])) 
    if isinstance(lista, list) else lista
)

#c. Normalizo nombre de actor principal.
#i. Unifico caracteres.
df_final["nombre_actor_principal"] = df_final["nombre_actor_principal"].apply(
    lambda x: normalizar_strings(x) if isinstance(x, str) else x
)
#ii. Unifico actores que se pueden escribir igual.
mapa_actores = {
    # Javier Milei
    "gobierno de javier milei": "javier milei",
    "gobierno nacional (javier milei)": "javier milei",
    "gobierno   javier milei": "javier milei",
    "gobierno (javier milei)": "javier milei",
    "gobierno nacional (presidente javier milei)": "javier milei",
    # Gobierno Nacional general
    "gobierno argentino": "gobierno nacional",
    # Banco Central
    "banco central": "banco central de la republica argentina bcra",
    "banco central de la republica argentina": "banco central de la republica argentina bcra",
    "banco central de la republica argentina (bcra)": "banco central de la republica argentina bcra",
    "bcra": "banco central de la republica argentina bcra",
    "banco central (santiago bausili)": "banco central de la republica argentina bcra)",
    # ANSES
    "anses": "administracion nacional de la seguridad social",
    # FMI / Fondo Monetario
    "fmi": "fondo monetario internacional",
    # Cristina Kirchner
    "cristina kirchner": "cristina fernandez de kirchner",
    "cristina fernandez de kirchner": "cristina fernandez de kirchner",
    # Luis Caputo
    "luis \"toto\" caputo": "luis caputo",
    "ministerio de economia   luis caputo": "luis caputo",
    "ministerio de economia (luis caputo)": "luis caputo",
    # Otros actores grandes.",
    "banco de la nacion argentina": "banco nacion",

}
#iii. Aplico.
def unificar_actores(actor):
    if not isinstance(actor, str):
        return actor
    actor = mapa_actores.get(actor, actor)  # si está en el mapa, reemplaza
    return actor

df_final["nombre_actor_principal"] = df_final["nombre_actor_principal"].apply(unificar_actores)


#d. Normalizo empresas mencionadas.
#i. Unifico caracteres.
df_final["empresas_mencionadas"] = df_final["empresas_mencionadas"].apply(
    lambda lista: [normalizar_strings(s) for s in lista] if isinstance(lista, list) else lista
)

mapa_empresas = {
    # Javier Milei
    "bcra": "banco central de la republica argentina bcra",
    "banco central bcra": "banco central de la republica argentina bcra",
    "banco central": "banco central de la republica argentina bcra",
    "banco central de la republica argentina (bcra)": "banco central de la republica argentina bcra",
    "banco nacion": "banco nacion",
    "banco de la nacion argentina": "banco nacion",

}

def unificar_empresas(lista):
    if isinstance(lista, list):
        return [mapa_empresas.get(s, s) for s in lista]
    return lista

#3. Aplico.
df_final["empresas_mencionadas"] = df_final["empresas_mencionadas"].apply(unificar_empresas)






In [7]:
#7. Analizo los valores únicos por columna.
#i. Sectores mencionales.
todos_los_sectores = [s for lista in df_final["sectores_mencionados"] for s in lista]
sectores_unicos = sorted(set(todos_los_sectores))
print("La cantidad de sectores post-procesamiento es de {}".format(len(sectores_unicos)))
#ii. Empresas mencionadas.
todas_las_empresas = [s for lista in df_final["empresas_mencionadas"] for s in lista]
empresas_unicas = sorted(set(todas_las_empresas))
print("La cantidad de empresas es de {}".format(len(empresas_unicas)))
#iii. Tickers mencionados.
todos_los_tickers = [s for lista in df_final["tickers_mencionados"] for s in lista]
tickers_unicos = sorted(set(todos_los_tickers))
print("La cantidad de tickers es de {}".format(len(tickers_unicos)))
#iv. Tipo de actor principal.
tipo_actor_principal_unicos = df_final["tipo_actor_principal"].unique()
print("La cantidad de tipo de actores principales es de {}".format(len(tipo_actor_principal_unicos)))
#v. Actor principal.
actor_principal = df_final["nombre_actor_principal"].unique()
print("La cantidad de actores principales es de {}".format(len(actor_principal)))
#vi. Caracter.
caracters_unicos = df_final["caracter"].unique()
print("La cantidad de caracteres es de {}".format(len(caracters_unicos)))
#vii. Shock.
shock_unicos = df_final["shock"].unique()
print("La cantidad de shock es de {}".format(len(shock_unicos)))
#viii. Tipo Evento.
tipo_evento_unicos = df_final["tipo_evento"].unique()
print("La cantidad de tipo de evento es de {}".format(len(tipo_evento_unicos)))

La cantidad de sectores post-procesamiento es de 22
La cantidad de empresas es de 8248
La cantidad de tickers es de 456
La cantidad de tipo de actores principales es de 12
La cantidad de actores principales es de 3019
La cantidad de caracteres es de 4
La cantidad de shock es de 5
La cantidad de tipo de evento es de 12


In [8]:
## Para las listas de empresas agarro las 50 que más aparecen y con esas hago dummies
# con el resto me quedo solo con la primera de la lista

# 1. top 50
top50 = (
    df_final.explode("empresas_mencionadas")["empresas_mencionadas"]
    .value_counts()
    .head(50)
    .index.tolist()
)


# 2. columnas dummy
for emp in top50:
    df_final[f"emp_{emp}"] = df_final["empresas_mencionadas"].apply(
        lambda lista: int(emp in lista if isinstance(lista, list) else False)
    )

# 3. otras
def primera_otras(lista):
    if not isinstance(lista, list) or len(lista)==0:
        return None
    for e in lista:
        if e not in top50:
            return e
    return None

df_final["empresa_otras"] = df_final["empresas_mencionadas"].apply(primera_otras)


In [9]:
from rapidfuzz import process, fuzz
import os
import json
import re
import pandas as pd
import unicodedata

def generar_diccionario_sugerencias(
    lista_elementos,
    nombre_salida="sugerencias",
    umbral=75,
    limite=10,
    min_longitud=7,
    filtrar_multiples = False
):
    """
    Genera un diccionario de normalización basado en similitud textual (RapidFuzz).
      ✅ Ignora palabras cortas (≤ min_longitud - 1)
      ✅ Elimina stopwords genéricas
      ✅ Ignora frases geográficas comunes (p.ej. "provincia de buenos aires")
      ✅ Exige similitud de palabras (Jaccard >= 0.4) y >= 2 palabras exactas en común
      ✅ Evita cruzar entidades de distinto dominio ("banco", "partido", "ministerio", etc.)
      ✅ Prioriza nombres sin paréntesis
      ✅ Elimina duplicados, aplica transitividad, y ordena alfabéticamente
      Si filtrar_multiple, entonces intenta quedarse con un solo actor cuando hay varios
    """

    print(f"🧩 Generando diccionario para '{nombre_salida}'...")

    # --- Configuración ---
    STOPWORDS = {
        "administracion",  "grupo", "sociedad", "compania", "compañia",
        "canal", "plc", "sa", "s.a", "s.a.", "sac", "corp", "company", "inc",
        "holding", "cooperativa", "cooperativo", "union", "universidad", "capital", "ventures", "holding", "ltd", "news",
        "instituto", "argentina", "grupo", "fundacion", "federacion", "ente", "empresa", "investment", "direccion",
        "departamento", "consultora", "corporacion", "consejo", "consorcio", "confederacion", "camara", "comision", "centro", "group",
        "nacional", "decreto", "justicia", "federal", "ministerio","mesa", "enlace", "municipalidad", "municipio", "oficina",
        "productores", "bolsa", "markets", "transportadora", "secretaria", "mercado", "sindicato", "supermercados"


    }


    FRASES_COMUNES = [
        "provincia de ",
        "ciudad de ",
        "republica argentina",
        " de valores",                  
        "mercado bursatil ",  
        "union industrial ",
        "union argentina ",
        "asset management", "corte suprema de justicia de", 
        "gobierno de estados unidos ", 
        "gobierno de"
        "tribunal oral "
        "banco de "
    ]

    # no mezcla palabras de diferentes dominios
    DOMINIOS = {
        "banco", "bolsa", "mercado", "caja", "comision", "ministerio",
        "partido", "universidad", "secretaria",
        "policia", "direccion", "legislatura",
        "camara", "federacion", 
        "asociacion", "empresa", "cooperativa", "corte suprema de justicia de ", 
        "tribunal oral ", "union", "karina ", "javier ", "banco galicia ", "corea", "sindicato", "supermercados", "brasil"
    }

    # --- FUNCIONES AUXILIARES ---
    def limpiar_stopwords(texto):
        palabras = [p for p in texto.split() if p.lower() not in STOPWORDS]
        return " ".join(palabras)

    def limpiar_frases_comunes(texto):
        texto = texto.lower()
        for frase in FRASES_COMUNES:
            texto = texto.replace(frase, "")
        return texto.strip()

    def palabras_en_comun(a, b):
        set_a = set(a.split())
        set_b = set(b.split())
        return len(set_a & set_b)

    def similitud_palabras(a, b):
        set_a, set_b = set(a.split()), set(b.split())
        if not set_a or not set_b:
            return 0
        inter = len(set_a & set_b)
        union = len(set_a | set_b)
        return inter / union  # índice de Jaccard

    def tipo_entidad(nombre):
        nombre_lower = nombre.lower()
        for d in DOMINIOS:
            if d in nombre_lower:
                return d
        return "otro"


    def tomar_primer_actor(texto):
        # si hay 2 actores quedarme con el primero
        return re.split(r"\b(?:y|&|con)\b|\s{3,}", texto, maxsplit=1)[0].strip()
    

    # --- 1. Contar frecuencias ---
    conteo = pd.Series(lista_elementos).value_counts()
    items = [i for i in conteo.index if len(i.strip()) >= min_longitud]
    print(f"Filtradas {len(conteo) - len(items)} entradas cortas (≤ {min_longitud - 1} caracteres).")

    # --- 2. Buscar coincidencias ---
    pairs = {}
    for e in items:
        e_original = e
        if filtrar_multiples:
            e = tomar_primer_actor(e)
        e_clean = limpiar_stopwords(limpiar_frases_comunes(e))
        similares = process.extract(
            e_clean, [limpiar_stopwords(limpiar_frases_comunes(i)) for i in items],
            scorer=fuzz.token_sort_ratio, limit=limite
        )

        tipo_e = tipo_entidad(e)

        for s_clean, score, idx in similares:
            s = items[idx]
            if s == e or score < umbral:
                continue

            tipo_s = tipo_entidad(s)
            if tipo_e != tipo_s and not (tipo_e == "otro" or tipo_s == "otro"):
                # evitar cruces entre dominios distintos
                continue

            if palabras_en_comun(e, s) < 2:
                continue
            if similitud_palabras(e, s) < 0.4:
                continue

            # Clave ordenada para evitar duplicados
            key = tuple(sorted([e, s]))

            # Priorizar el que no tiene paréntesis
            freq_e = conteo.get(e, 0)
            freq_s = conteo.get(s, 0)
            canonico = e if freq_e >= freq_s else s
            variante = s if canonico == e else e

            if key not in pairs or score > pairs[key]["score"]:
                pairs[key] = {"variante": variante, "canonico": canonico, "score": score}

    # --- 3. Crear diccionario plano ---
    diccionario = {r["variante"]: r["canonico"] for r in pairs.values()}

    # --- 4. Resolver transitividad ---
    def resolver_transitividad(dic):
        def find_root(x, seen=None):
            if seen is None:
                seen = set()
            if x not in dic or x in seen:
                return x
            seen.add(x)
            return find_root(dic[x], seen)

        resultado = {}
        for k, v in dic.items():
            root = find_root(v)
            resultado[k] = root
        return resultado

    diccionario_final = resolver_transitividad(diccionario)

    # --- 5. Ordenar alfabéticamente ---
    diccionario_ordenado = dict(
        sorted(diccionario_final.items(), key=lambda x: (x[1].lower(), x[0].lower()))
    )

    # --- 6. Guardar resultado ---
    carpeta_salida = "./dicts"
    os.makedirs(carpeta_salida, exist_ok=True)

    output_path = os.path.join(carpeta_salida, f"sugerencias_{nombre_salida}.json")
    with open(output_path, "w", encoding="utf8") as f:
        json.dump(diccionario_ordenado, f, ensure_ascii=False, indent=2)

    print(f"✅ Diccionario '{nombre_salida}' guardado en {output_path} con {len(diccionario_ordenado)} entradas.")
    return diccionario_ordenado

In [10]:
# Genero diccionario buscando similares con la logica de generar_diccionario_sugerencias
dicc_empresas = generar_diccionario_sugerencias(df_final["empresa_otras"].unique(), nombre_salida="empresas")
dicc_actor = generar_diccionario_sugerencias(df_final["nombre_actor_principal"].unique(), nombre_salida="actor_principal", filtrar_multiples=True)




🧩 Generando diccionario para 'empresas'...
Filtradas 410 entradas cortas (≤ 6 caracteres).
✅ Diccionario 'empresas' guardado en ./dicts/sugerencias_empresas.json con 231 entradas.
🧩 Generando diccionario para 'actor_principal'...
Filtradas 229 entradas cortas (≤ 6 caracteres).
✅ Diccionario 'actor_principal' guardado en ./dicts/sugerencias_actor_principal.json con 648 entradas.


In [11]:
def aplicar_mapa(df, columna, mapa):
    """
    Reemplaza valores en strings o listas dentro de columna
    usando diccionario exacto {variante: canonico} sin normalizar.
    """
    if columna not in df.columns:
        raise ValueError(f"La columna '{columna}' no existe en el DataFrame.")

    def mapear_uno(x):
        if x is None:
            return x
        # si es string
        if isinstance(x, str):
            return mapa.get(x, x)
        # si es lista / tuple / set
        if isinstance(x, (list, tuple, set)):
            t = type(x)
            mapped = [mapear_uno(e) for e in x]
            if t is set: return set(mapped)
            if t is tuple: return tuple(mapped)
            return mapped   # list
        return x

    out = df.copy()
    out[columna] = out[columna].apply(mapear_uno)
    return out

df_final = aplicar_mapa(df_final, "empresa_otras", dicc_empresas)
df_final = aplicar_mapa(df_final, "nombre_actor_principal", dicc_actor)


In [12]:
# Habiendo ya unificado algo los actores, tomo el top 50, hago dummies y los remuevo de los string originales

# 1) separador robusto para partir actores dentro del string original
pat_split = re.compile(r"\s{3,}|\s+y\s+|[,;/]")

def split_actores(val: str):
    if not isinstance(val, str) or not val.strip():
        return []
    partes = pat_split.split(val)
    return [p.strip() for p in partes if p.strip()]

# 2) top 50 actores (a partir del string, sin normalizar)
df_final["__actores_lista"] = df_final["nombre_actor_principal"].apply(split_actores)
top50_actores = (
    df_final["__actores_lista"].explode().value_counts().head(50).index.tolist()
)

# 3) regex para detectar y eliminar (ordenar por longitud desc evita matches parciales primero)
actores_ordenados = sorted(top50_actores, key=len, reverse=True)
alternativas = "|".join(re.escape(a) for a in actores_ordenados)
# palabra/frase completa: límites de palabra a ambos lados (maneja multi-palabra igual)
pat_remove = re.compile(rf"\b(?:{alternativas})\b")

# 4) dummies por presencia (match como palabra/frase completa en el string)
def has_actor(s: str, actor: str) -> int:
    if not isinstance(s, str) or not s.strip():
        return 0
    return int(re.search(rf"\b{re.escape(actor)}\b", s) is not None)

# nombres de columna seguros
def slug(s: str, maxlen=60):
    s2 = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s2 = re.sub(r"[^A-Za-z0-9]+", "_", s2).strip("_").lower()
    return s2[:maxlen]

for actor in top50_actores:
    col = f"act_{slug(actor)}"
    df_final[col] = df_final["nombre_actor_principal"].apply(lambda s: has_actor(s, actor))

# 5) quitar del string TODOS los actores del top50, dejando el resto
def quitar_top_en_string(s: str):
    if not isinstance(s, str) or not s.strip():
        return None
    # quitar frases/palabras del top
    out = pat_remove.sub(" ", s)
    # limpiar conectores/residuos: múltiples espacios, " y " sueltos, comas dobles, etc.
    out = re.sub(r"\s{3,}|\s*,\s*,+", " ", out)        # espacios extra / comas repetidas
    out = re.sub(r"\s+y\s+", " ", out)                 # ' y ' si quedó flotando
    out = re.sub(r"\s{2,}", " ", out).strip(" ,;/\t")  # colapsar espacios y recortar
    return out if out else None

df_final["actor_principal_sin_top"] = df_final["nombre_actor_principal"].apply(quitar_top_en_string)

# limpiar columna auxiliar
df_final.drop(columns=["__actores_lista"], inplace=True)




In [13]:
top50_actores

['javier milei',
 'desconocido',
 'banco central de la republica argentina bcra',
 'gobierno nacional',
 'donald trump',
 'administracion nacional de la seguridad social anses',
 'luis caputo',
 'fondo monetario internacional fmi',
 'gobierno',
 'indec',
 'banco nacion',
 'lemon',
 'ministerio de economia',
 'axel kicillof',
 'patricia bullrich',
 'jorge macri',
 'control aduanero arca',
 'agencia de recaudacion',
 'gobierno de milei',
 'camara de diputados',
 'guillermo francos',
 'gobierno de la ciudad de buenos aires',
 'mauricio macri',
 'casa rosada',
 'cristina fernandez de kirchner',
 'ypf',
 'federico sturzenegger',
 'banco provincia',
 'manuel adorni',
 'cgt',
 'mercado agroganadero de canuelas',
 'victoria villarruel',
 'mercado pago',
 'presidente javier milei',
 'estados unidos',
 'pro',
 'corte suprema',
 'cat cafe buenos aires',
 'reserva federal eeuu',
 'gobierno de trump',
 'la libertad avanza',
 'poder ejecutivo',
 'ministerio de economia luis caputo',
 'karina milei',

In [14]:
#c. Cuáles son los tickers con mayor mención?
conteo_tickers = pd.Series(todos_los_tickers).value_counts()
conteo_tickers

AL30     349
BTC      330
LIBRA    296
ETH      290
SOL      165
        ... 
EMB        1
TG25       1
VT         1
ENRON      1
RYAN       1
Name: count, Length: 456, dtype: int64

In [15]:
#9. Creo nuevas columnas booleanas con True o False según aparición.
top_tickers = list(conteo_tickers.head(10).keys())

for ticker in top_tickers:
    df_final[f'ticker_{ticker}'] = df_final['tickers_mencionados'].apply(lambda x: ticker in x)

for sector in sectores_unicos:
    df_final[f'sector_{sector}'] = df_final['sectores_mencionados'].apply(lambda x: sector in x)

for actor in tipo_actor_principal_unicos:
    df_final[f'tipo_actor_principal_{actor}'] = df_final['tipo_actor_principal'].apply(lambda x: actor in x)

for caracter in caracters_unicos:
    df_final[f'caracter_{caracter}'] = df_final['caracter'].apply(lambda x: caracter in x)

for shock in shock_unicos:
    df_final[f'shock_{shock}'] = df_final['shock'].apply(lambda x: shock in x)

for tipo_evento in tipo_evento_unicos:
    df_final[f'tipo_evento_{tipo_evento}'] = df_final['tipo_evento'].apply(lambda x: tipo_evento in x)


/tmp/ipykernel_108958/129312033.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final[f'tipo_actor_principal_{actor}'] = df_final['tipo_actor_principal'].apply(lambda x: actor in x)
/tmp/ipykernel_108958/129312033.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final[f'caracter_{caracter}'] = df_final['caracter'].apply(lambda x: caracter in x)
/tmp/ipykernel_108958/129312033.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor 

In [16]:
#10. Nuevas columnas con extensión de la lista.
df_final['cant_empresas'] = df_final['empresas_mencionadas'].apply(len)
df_final['cant_tickers'] = df_final['tickers_mencionados'].apply(len)
df_final['cant_sectores'] = df_final['sectores_mencionados'].apply(len)

/tmp/ipykernel_108958/2473905255.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['cant_empresas'] = df_final['empresas_mencionadas'].apply(len)
/tmp/ipykernel_108958/2473905255.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['cant_tickers'] = df_final['tickers_mencionados'].apply(len)
/tmp/ipykernel_108958/2473905255.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once

In [17]:
#11. Nuevas columnas con la extensión de la noticia.
df_final['palabras_titulo'] = df_final['titulo'].apply(lambda x: len(str(x).split()))
df_final['palabras_noticia'] = df_final['contenido'].apply(lambda x: len(str(x).split()))

/tmp/ipykernel_108958/272152380.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['palabras_titulo'] = df_final['titulo'].apply(lambda x: len(str(x).split()))
/tmp/ipykernel_108958/272152380.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['palabras_noticia'] = df_final['contenido'].apply(lambda x: len(str(x).split()))


In [18]:
#12. Creo columna booleana con el nombre del diario y la sección.
diarios_unicos = df_final["diario"].unique()
secciones_unicas = df_final["seccion"].unique()

for diario in diarios_unicos:
    df_final[f'diario_{diario}'] = df_final['diario'].apply(lambda x: diario in x)

#for seccion in secciones_unicas:
#    df_final[f'seccion_{seccion}'] = df_final['seccion'].apply(lambda x: seccion in x)

/tmp/ipykernel_108958/2296555431.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final[f'diario_{diario}'] = df_final['diario'].apply(lambda x: diario in x)
/tmp/ipykernel_108958/2296555431.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final[f'diario_{diario}'] = df_final['diario'].apply(lambda x: diario in x)
/tmp/ipykernel_108958/2296555431.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all

In [19]:
#13. Elimino las columnas que ya no voy a tener en cuenta.
#a. Listado de columnas a eliminar.
columnas_a_eliminar = [
    'diario', 
    'titulo', 
    'contenido', 
    'url', 
    'seccion', 
    'tipo_actor_principal', 
    'nombre_actor_principal', 
    'empresas_mencionadas',
    'tickers_mencionados',
    'sectores_mencionados',
    'tipo_evento',
    'shock',
    'caracter'
]

#b. Eliminación.
df_final = df_final.drop(columns=columnas_a_eliminar)

In [20]:
#14. Exporto.
df_final = df_final.sort_values(by="fecha",ascending=True).reset_index(drop=True)
df_final.to_csv(df_exportacion_path,index=False)

In [21]:
pd.Series(df_final.columns).to_csv("columnas_df_final.csv", index=False)
